# 7교시. 추출 결과 검증 및 데이터 저장

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leecks1119/document_ai_lecture/blob/master/colab/07_validation_export.ipynb)

**이번 교시 행동:** 오류·경고·사람 검토를 분리하고, 공개된 승인 정답 경로에서만 Excel을 만듭니다.

**통과 증거:** `course_outputs/receipt_result.xlsx`

> Google Colab도 외부 클라우드입니다. 조직 승인 없는 개인·회사 문서는
> 업로드하지 않습니다. 필수 실습은 저장소의 비식별 공개·합성 샘플만
> 사용합니다.

화면의 **처리 방식**을 먼저 확인합니다.

- **지금 이 사진을 직접 읽었습니다:** 현재 파일에 OCR 모델을 실행한 결과입니다.
- **수업용 예제 결과를 불러왔습니다:** 현재 파일을 분석한 결과가 아닙니다.
- 3분 이상 멈추면 실행을 중지하고 수업용 예제로 계속합니다.
- 각 교시 끝에서 `CHECKPOINT PASS`와 산출물 파일을 확인합니다.


## 이 노트북에서 내가 하는 일

- **필수 실습:** 잘못된 값은 저장을 막고, 원본 확인과 승인 뒤에만 Excel을 내려받습니다.
- **내가 바꾸는 곳:** 승인 여부·검토자·원본 확인 메모 세 곳만 채웁니다.
- **인터넷 자료로 다시 실험:** 내 실험 문서에서 틀린 값 하나를 일부러 넣어 어떤 규칙이 저장을 막아야 하는지 확인합니다.

먼저 제공 샘플로 끝까지 실행해 `CHECKPOINT PASS`를 만드세요. 그다음
[공개·비식별 실습 자료 찾기](https://github.com/leecks1119/document_ai_lecture/blob/master/docs/public_practice_sources.md)를 보고
입력 한 장만 바꾸어 다시 실행합니다. 2교시에서 고른 자료와 결과 파일은
3~7교시에 그대로 이어 쓰므로 매 시간 새 자료를 찾을 필요가 없습니다.

> `🟢 그대로 실행하는 셀`은 수정하지 않습니다. `🟠 내가 짧게 바꾸는
> 셀`만 필수이고, `🔵 원하면 바꾸는 셀`은 시간이 남을 때 합니다.
> 정답은 모두 공개되어 있으므로 정답을 먼저 복사하고 결과를 관찰해도 됩니다.

## 코드 셀을 읽는 방법

각 코드 셀의 맨 위에는 `코드 읽기` 주석이 있습니다.

1. `수정하지 않습니다`라고 적힌 셀은 설명을 읽고 그대로 실행합니다.
2. 주황색 필수 `TODO`만 채웁니다. 파란색 선택 `TODO`는 건너뛰어도 됩니다.
3. 실행 출력에서 `코드 읽는 법`과 `확인할 결과`를 다시 확인합니다.
4. `단계 실행 완료`가 나온 뒤 다음 코드 셀로 이동합니다.

Python 문법 전체를 먼저 이해할 필요는 없습니다. 변수에 어떤 값이 들어가고,
실행 뒤 어떤 결과가 달라지는지를 중심으로 읽습니다.


In [ ]:
def _show_learning_message(markdown_text):
    try:
        from IPython.display import Markdown, display
        display(Markdown(markdown_text))
    except ImportError:
        print(markdown_text)


def show_lab_step(
    current,
    total,
    title,
    action,
    expected,
    code_help,
    edit_kind,
):
    cell_kind = {
        "required": "🟠 내가 짧게 바꾸는 셀",
        "optional": "🔵 원하면 바꾸는 셀",
        "none": "🟢 그대로 실행하는 셀",
    }[edit_kind]
    _show_learning_message(
        f"""---
### {cell_kind} · {current}/{total} · {title}

**지금 할 일:** {action}

**코드 읽는 법:** {code_help}

**이 단계에서 확인할 결과:** {expected}
"""
    )


def complete_lab_step(current, total, expected):
    next_action = (
        "결과를 확인한 뒤 다음 코드 셀을 실행하세요."
        if current < total
        else "마지막 CHECKPOINT와 산출물 파일을 확인하세요."
    )
    _show_learning_message(
        f"""> ✅ **{current}/{total} 단계 실행 완료**
>
> **결과 확인:** {expected}
>
> **다음 행동:** {next_action}
"""
    )

# ── 코드 읽기 ─────────────────────────────────────────────
# 검증 결과와 Excel을 저장할 공통 폴더·파일 인계 함수를 준비합니다. 설정 코드이므로 수정하지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(1, 12, '공통 환경 준비', '검증 결과와 Excel을 저장할 공통 환경을 준비합니다.', 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.', '검증 결과와 Excel을 저장할 공통 폴더·파일 인계 함수를 준비합니다. 설정 코드이므로 수정하지 않습니다.', 'none')

import json
import os
import platform
import sys
from pathlib import Path

OUTPUT_DIR = Path("course_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
VALIDATION_MODE = os.getenv("COURSE_VALIDATE_EXAMPLE") == "1"

def upload_previous_artifact(filename):
    target = OUTPUT_DIR / filename
    if target.exists() or VALIDATION_MODE:
        return target if target.exists() else None
    try:
        from google.colab import files
    except ImportError:
        return None
    print(f"이전 교시에서 내려받은 {filename}을 선택하세요.")
    uploaded = files.upload()
    if filename not in uploaded:
        raise FileNotFoundError(
            f"{filename}이 선택되지 않았습니다. 준비 입력을 쓰려면 "
            "USE_COURSE_EXAMPLE=True로 바꾸세요."
        )
    target.write_bytes(uploaded[filename])
    return target


def download_artifact(path):
    if VALIDATION_MODE:
        return
    try:
        from google.colab import files
    except ImportError:
        return
    files.download(str(path))

print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("공통 작업 폴더:", OUTPUT_DIR.resolve())

COURSE_ASSET_BASE_URL = (
    "https://raw.githubusercontent.com/leecks1119/"
    "document_ai_lecture/master/"
)

def load_course_assets(*relative_paths):
    if VALIDATION_MODE:
        local_root = os.getenv("COURSE_LOCAL_ASSET_ROOT")
        if not local_root:
            raise RuntimeError(
                "자동 검증용 COURSE_LOCAL_ASSET_ROOT가 필요합니다."
            )
        root = Path(local_root)
        return {
            path: (root / path).read_bytes()
            for path in relative_paths
        }

    import requests

    loaded = {}
    missing = []
    for path in relative_paths:
        try:
            response = requests.get(
                COURSE_ASSET_BASE_URL + path,
                timeout=30,
            )
            response.raise_for_status()
            loaded[path] = response.content
        except requests.RequestException as exc:
            print(f"자동 다운로드 실패: {Path(path).name} · {exc}")
            missing.append(path)

    if missing:
        from google.colab import files

        expected = ", ".join(Path(path).name for path in missing)
        print("다음 파일을 저장소에서 내려받아 선택하세요:", expected)
        uploaded = files.upload()
        uploaded_by_name = {
            Path(name).name: content
            for name, content in uploaded.items()
        }
        for path in missing:
            filename = Path(path).name
            if filename not in uploaded_by_name:
                raise FileNotFoundError(
                    f"{filename}이 선택되지 않았습니다."
                )
            loaded[path] = uploaded_by_name[filename]

    return loaded

complete_lab_step(1, 12, 'Python·Platform·공통 작업 폴더가 표시되어야 합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# 최종 앱에서 사용할 Streamlit 버전을 확인하고 필요할 때만 설치합니다. 버전 숫자는 수업 중 임의로 바꾸지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(2, 12, 'Streamlit 준비', '최종 앱에 필요한 Streamlit 버전을 확인하고 준비합니다.', '오류 없이 끝나면 최종 앱 실행 환경이 준비된 것입니다.', '최종 앱에서 사용할 Streamlit 버전을 확인하고 필요할 때만 설치합니다. 버전 숫자는 수업 중 임의로 바꾸지 않습니다.', 'none')

import importlib.metadata
import subprocess

required_streamlit = "1.60.0"
try:
    installed_streamlit = importlib.metadata.version("streamlit")
except importlib.metadata.PackageNotFoundError:
    installed_streamlit = None
if installed_streamlit != required_streamlit:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", f"streamlit=={required_streamlit}"]
    )

complete_lab_step(2, 12, '오류 없이 끝나면 최종 앱 실행 환경이 준비된 것입니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `find_spec()`로 openpyxl 설치 여부를 확인합니다. 없을 때만 설치해 Excel 생성 함수를 사용할 수 있게 합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(3, 12, 'Excel 라이브러리 준비', 'Excel 생성과 재검사에 필요한 openpyxl을 준비합니다.', '오류 없이 끝나면 Excel 기능을 사용할 수 있습니다.', '`find_spec()`로 openpyxl 설치 여부를 확인합니다. 없을 때만 설치해 Excel 생성 함수를 사용할 수 있게 합니다.', 'none')

import importlib.util
import subprocess
if importlib.util.find_spec("openpyxl") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "openpyxl==3.1.5"]
    )
from copy import deepcopy
from datetime import date
from openpyxl import Workbook, load_workbook

complete_lab_step(3, 12, '오류 없이 끝나면 Excel 기능을 사용할 수 있습니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `GOLDEN_RECEIPT`는 검증 규칙과 Excel 구조를 확인할 공개 정답입니다. 이 셀은 데이터를 등록하므로 수정하지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(4, 12, '검증용 정답 데이터 준비', '영수증 원문·품목·원문 근거가 포함된 정답을 등록합니다.', '오류 없이 끝나면 검증용 데이터가 준비된 것입니다.', '`GOLDEN_RECEIPT`는 검증 규칙과 Excel 구조를 확인할 공개 정답입니다. 이 셀은 데이터를 등록하므로 수정하지 않습니다.', 'none')

GOLDEN_OCR_TEXT = '이태리집\n거래일시 2025-10-04 12:33:37\n페퍼로니 앤 치즈 29,000 1 29,000\n토마토 파스타 14,000 1 14,000\n수제 돈가스 13,000 1 13,000\n새우 칠리치 필라 14,000 1 14,000\n콜라 2,000 3 6,000\n합계 금액 76,000\n부가세 과세물품가액 69,094\n부가세 6,906\n'
GOLDEN_VLM_MARKDOWN = '# 이태리집\n\n> **수업용 VLM 구조 예제** — 지금 모델을 실행해 만든 결과가 아닙니다.\n\n거래일시: 2025-10-04 12:33:37\n\n| 품목 | 수량 | 단가 | 금액 |\n| --- | ---: | ---: | ---: |\n| 페퍼로니 앤 치즈 | 1 | 29,000원 | 29,000원 |\n| 토마토 파스타 | 1 | 14,000원 | 14,000원 |\n| 수제 돈가스 | 1 | 13,000원 | 13,000원 |\n| 새우 칠리치 필라 | 1 | 14,000원 | 14,000원 |\n| 콜라 | 3 | 2,000원 | 6,000원 |\n\n**합계: 76,000원**\n\n부가세 과세물품가액 69,094\n부가세 6,906\n'
GOLDEN_RECEIPT = {'document_type': 'receipt',
 'store_name': '이태리집',
 'date': '2025-10-04',
 'total_amount': 76000,
 'items': [{'name': '페퍼로니 앤 치즈',
            'quantity': 1,
            'unit_price': 29000,
            'line_total': 29000},
           {'name': '토마토 파스타', 'quantity': 1, 'unit_price': 14000, 'line_total': 14000},
           {'name': '수제 돈가스', 'quantity': 1, 'unit_price': 13000, 'line_total': 13000},
           {'name': '새우 칠리치 필라',
            'quantity': 1,
            'unit_price': 14000,
            'line_total': 14000},
           {'name': '콜라', 'quantity': 3, 'unit_price': 2000, 'line_total': 6000}],
 'adjustments': {'discount': 0, 'tax': 0, 'service': 0, 'rounding': 0},
 'tax_breakdown': {'mode': 'included_in_item_prices',
                   'supply_amount': 69094,
                   'vat': 6906,
                   'payable_total': 76000},
 'raw_values': {'store_name': '이태리집',
                'date': '2025-10-04 12:33:37',
                'total_amount': '76,000'},
 'cleaned_values': {'store_name': '이태리집', 'date': '2025-10-04', 'total_amount': 76000},
 'evidence': {'store_name': {'raw_value': '이태리집', 'line': 1},
              'date': {'raw_value': '거래일시 2025-10-04 12:33:37', 'line': 2},
              'total_amount': {'raw_value': '합계 금액 76,000', 'line': 8}},
 'source_mode': 'course_example_rule_extraction'}

complete_lab_step(4, 12, '오류 없이 끝나면 검증용 데이터가 준비된 것입니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `USE_COURSE_EXAMPLE=True`는 공개 입력, `False`는 4교시 JSON 업로드입니다. `validate_receipt()`
# 결과의 오류와 경고를 확인합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(5, 12, '이전 교시 결과 불러오기', '이전 JSON을 읽거나 공개 복구 입력으로 전환합니다.', '입력 모드와 검증 결과가 표시되어야 합니다.', '`USE_COURSE_EXAMPLE=True`는 공개 입력, `False`는 4교시 JSON 업로드입니다. `validate_receipt()` 결과의 오류와 경고를 확인합니다.', 'none')

input_path = OUTPUT_DIR / "receipt.json"
# 기본값 True: 새 Colab에서도 공개 준비 입력으로 바로 실행합니다.
# 앞 교시 파일을 이어 쓰려면 False로 바꾸고 업로드 창에서 선택합니다.
USE_COURSE_EXAMPLE = True
if not input_path.exists() and not USE_COURSE_EXAMPLE:
    upload_previous_artifact("receipt.json")
if input_path.exists():
    receipt = json.loads(input_path.read_text(encoding="utf-8"))
    INPUT_MODE = "PREVIOUS_LESSON"
else:
    receipt = deepcopy(GOLDEN_RECEIPT)
    INPUT_MODE = "COURSE_EXAMPLE"


def validate_receipt(data):
    warnings, errors = [], []
    for field in ("store_name", "date", "total_amount", "items"):
        if data.get(field) in (None, "", []):
            errors.append(f"필수값 누락: {field}")
    try:
        parsed_date = date.fromisoformat(data.get("date", ""))
        if parsed_date > date.today():
            warnings.append("미래 날짜입니다. 원본을 확인하세요.")
    except ValueError:
        errors.append("date는 YYYY-MM-DD 형식이어야 합니다.")

    total = data.get("total_amount")
    if isinstance(total, bool) or not isinstance(total, int) or total < 0:
        errors.append("total_amount는 0 이상의 정수여야 합니다.")
    item_sum = 0
    for index, item in enumerate(data.get("items") or [], start=1):
        values = [item.get(key) for key in ("quantity", "unit_price", "line_total")]
        if not all(isinstance(value, int) and not isinstance(value, bool) for value in values):
            errors.append(f"{index}번째 품목 금액 형식 오류")
            continue
        if values[0] * values[1] != values[2]:
            errors.append(f"{index}번째 품목 수량×단가 오류")
        item_sum += values[2]
    adjustments = data.get("adjustments") or {}
    expected = (
        item_sum
        - adjustments.get("discount", 0)
        + adjustments.get("tax", 0)
        + adjustments.get("service", 0)
        + adjustments.get("rounding", 0)
    )
    if isinstance(total, int) and not isinstance(total, bool) and expected != total:
        errors.append(f"품목·조정 후 합계 {expected:,}원과 총액 {total:,}원이 다릅니다.")
    tax_breakdown = data.get("tax_breakdown")
    if tax_breakdown and tax_breakdown.get("mode") == "included_in_item_prices":
        supply = tax_breakdown.get("supply_amount")
        vat = tax_breakdown.get("vat")
        payable = tax_breakdown.get("payable_total")
        if not all(isinstance(value, int) and not isinstance(value, bool)
                   for value in (supply, vat, payable)):
            errors.append("포함세액 내역은 정수 금액이어야 합니다.")
        elif supply + vat != payable or payable != total:
            errors.append("공급가액·포함 부가세·총액 관계가 맞지 않습니다.")
        if adjustments.get("tax", 0) != 0:
            errors.append("포함 부가세를 adjustments.tax에 다시 더하면 이중 계산됩니다.")
    for field in ("store_name", "date", "total_amount"):
        if not (data.get("evidence") or {}).get(field):
            warnings.append(f"{field}의 원본 근거가 없습니다.")
    return {"valid": not errors, "warnings": warnings, "errors": errors}


validation = validate_receipt(receipt)
source_text = receipt.get("source_text") or GOLDEN_OCR_TEXT
print(
    "입력 자료:",
    (
        "4교시에서 만든 JSON을 불러왔습니다."
        if INPUT_MODE == "PREVIOUS_LESSON"
        else "수업용 예제 JSON을 불러왔습니다."
    ),
)
if INPUT_MODE == "COURSE_EXAMPLE":
    print("중요: 지금 업로드한 문서를 새로 분석한 결과가 아닙니다.")
print("검증:", validation)
if not validation["valid"]:
    print(
        "BLOCKED_BY_VALIDATION: 아래 최종 앱에서 원본과 대조해 "
        "값을 수정한 뒤 승인하세요."
    )

complete_lab_step(5, 12, '입력 모드와 검증 결과가 표시되어야 합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `validate_receipt()`는 필수값·계산·근거를 검사하고 `save_reviewed_excel()`은 검증과 승인 조건이 모두 맞을 때만
# 세 시트를 만듭니다.
# ──────────────────────────────────────────────────────────
show_lab_step(6, 12, '검증·Excel 함수 준비', '오류·경고·승인 상태를 검사하고 Excel을 만드는 함수를 등록합니다.', '오류 없이 끝나면 검증과 저장 함수를 사용할 수 있습니다.', '`validate_receipt()`는 필수값·계산·근거를 검사하고 `save_reviewed_excel()`은 검증과 승인 조건이 모두 맞을 때만 세 시트를 만듭니다.', 'none')

def safe_text(value):
    if isinstance(value, str) and value.lstrip(" \t\r\n").startswith(
        ("=", "+", "-", "@")
    ):
        return "'" + value
    return value


def save_reviewed_excel(data, validation, review_record, output_path, source_text):
    if not validation["valid"]:
        return False
    if review_record.get("decision") not in {"APPROVED", "CHANGED"}:
        return False

    workbook = Workbook()
    summary = workbook.active
    summary.title = "검토_요약"
    summary.append([
        "field", "raw_value", "cleaned_value", "final_value",
        "decision", "reviewer", "reviewed_at", "change_reason",
    ])
    raw = data.get("raw_values") or {}
    cleaned = data.get("cleaned_values") or {}
    for field in ("store_name", "date", "total_amount"):
        summary.append([
            field,
            safe_text(raw.get(field)),
            safe_text(cleaned.get(field)),
            safe_text(data.get(field)),
            review_record["decision"],
            safe_text(review_record["reviewer"]),
            review_record["reviewed_at"],
            safe_text(review_record["note"]),
        ])

    items = workbook.create_sheet("품목")
    items.append(["품목", "수량", "단가", "금액"])
    for item in data["items"]:
        items.append([
            safe_text(item["name"]),
            item["quantity"],
            item["unit_price"],
            item["line_total"],
        ])

    evidence = workbook.create_sheet("원문_근거")
    evidence.append(["source_mode", data.get("source_mode")])
    evidence.append(["ocr_text", safe_text(source_text)])
    evidence.append(["evidence", safe_text(json.dumps(
        data.get("evidence") or {}, ensure_ascii=False
    ))])
    workbook.save(output_path)
    return True

complete_lab_step(6, 12, '오류 없이 끝나면 검증과 저장 함수를 사용할 수 있습니다.')


## 시나리오 A. 기본값은 차단

사람이 원본을 보기 전에는 결과가 유효해도 다운로드를 열지 않습니다.


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `PENDING_REVIEW` 상태로 `save_reviewed_excel()`을 호출합니다. 반환값이 `False`이고 파일이 없으면 미승인 저장
# 차단이 정상입니다.
# ──────────────────────────────────────────────────────────
show_lab_step(7, 12, '미승인 저장 차단 확인', '검토하지 않은 결과로 Excel이 생성되지 않는지 검사합니다.', '`DEFAULT_BLOCKED PASS`와 파일 미생성을 확인합니다.', '`PENDING_REVIEW` 상태로 `save_reviewed_excel()`을 호출합니다. 반환값이 `False`이고 파일이 없으면 미승인 저장 차단이 정상입니다.', 'none')

blocked_path = OUTPUT_DIR / "pending_review.xlsx"
PENDING_REVIEW = {
    "decision": "PENDING",
    "reviewer": "",
    "reviewed_at": "",
    "note": "",
}
assert not save_reviewed_excel(
    receipt, validation, PENDING_REVIEW, blocked_path, source_text
)
assert not blocked_path.exists()
print("DEFAULT_BLOCKED PASS: 미승인 Excel 없음")

complete_lab_step(7, 12, '`DEFAULT_BLOCKED PASS`와 파일 미생성을 확인합니다.')


## 시나리오 B. 내가 직접 남기는 승인 기록

원본의 상호명·날짜·품목·총액을 직접 대조한 뒤 세 곳을 채웁니다.


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `my_decision`, `my_reviewer`, `my_review_note` 세 곳만 입력합니다. 원본 대조를 끝낸 뒤 실제 검토 기록을
# 남기는 단계입니다.
# ──────────────────────────────────────────────────────────
show_lab_step(8, 12, '내 승인 기록 입력', '승인 결정·검토자 이름·원본 확인 내용을 직접 입력합니다.', '빈칸 안내 또는 내가 남긴 승인 기록이 표시되어야 합니다.', '`my_decision`, `my_reviewer`, `my_review_note` 세 곳만 입력합니다. 원본 대조를 끝낸 뒤 실제 검토 기록을 남기는 단계입니다.', 'required')

# TODO: 원본 대조 뒤 세 곳을 채우세요.
my_decision = None
my_reviewer = None
my_review_note = None
if None in (my_decision, my_reviewer, my_review_note):
    print("빈칸이 있습니다. 아래 전체 정답과 비교하세요.")

complete_lab_step(8, 12, '빈칸 안내 또는 내가 남긴 승인 기록이 표시되어야 합니다.')


<details>
<summary>힌트와 전체 정답 보기</summary>

값 수정이 없으면 `APPROVED`, 수정했다면 `CHANGED`입니다. 검토자와
무엇을 확인했는지도 기록합니다.
</details>


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `reviewed_receipt`에 공개 샘플의 원본 대조 정답을 적용하고 `REVIEW_RECORD`와 함께 Excel을 만듭니다. 내 자료는
# 내가 승인 기록을 채우기 전까지 저장하지 않습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(9, 12, '승인 후 Excel 생성', '공개 승인 경로로 재검증하고 Excel 세 시트를 생성합니다.', '`REVIEWED_APPROVED PASS`와 Excel 파일 경로를 확인합니다.', '`reviewed_receipt`에 공개 샘플의 원본 대조 정답을 적용하고 `REVIEW_RECORD`와 함께 Excel을 만듭니다. 내 자료는 내가 승인 기록을 채우기 전까지 저장하지 않습니다.', 'none')

from datetime import datetime, timedelta, timezone

KST = timezone(timedelta(hours=9))
IS_COURSE_SAMPLE = (
    receipt.get("date") == "2025-10-04"
    and receipt.get("total_amount") == 76000
    and len(receipt.get("items") or []) == 5
)
reviewed_receipt = deepcopy(receipt)
item_corrections = []
if IS_COURSE_SAMPLE:
    for item, answer_item in zip(
        reviewed_receipt["items"],
        GOLDEN_RECEIPT["items"],
    ):
        if item["name"] != answer_item["name"]:
            item_corrections.append({
                "OCR 판독": item["name"],
                "원본 대조 후": answer_item["name"],
            })
            item["name"] = answer_item["name"]
    if item_corrections:
        print("공개 샘플의 품목명 원본 대조 정답:")
        for correction in item_corrections:
            print(
                "-",
                correction["OCR 판독"],
                "→",
                correction["원본 대조 후"],
            )

learner_completed_review = None not in (
    my_decision,
    my_reviewer,
    my_review_note,
)
default_decision = "CHANGED" if item_corrections else "APPROVED"
REVIEW_RECORD = {
    "decision": (
        my_decision
        if learner_completed_review
        else default_decision if IS_COURSE_SAMPLE else "PENDING"
    ),
    "reviewer": (
        my_reviewer
        if learner_completed_review
        else "공개 정답" if IS_COURSE_SAMPLE else ""
    ),
    "reviewed_at": datetime.now(KST).isoformat(timespec="seconds"),
    "note": (
        my_review_note
        if learner_completed_review
        else (
            "공개 비식별 원본 대조 후 OCR 품목명 수정"
            if item_corrections
            else "공개 비식별 원본과 주요 필드 대조 완료"
        )
        if IS_COURSE_SAMPLE
        else ""
    ),
}
reviewed_validation = validate_receipt(reviewed_receipt)
output_path = OUTPUT_DIR / "receipt_result.xlsx"
excel_created = save_reviewed_excel(
    reviewed_receipt,
    reviewed_validation,
    REVIEW_RECORD,
    output_path,
    source_text,
)
if excel_created:
    saved = load_workbook(output_path)
    assert saved.sheetnames == ["검토_요약", "품목", "원문_근거"]
    assert saved["검토_요약"]["E2"].value in {"APPROVED", "CHANGED"}
    print("REVIEWED_APPROVED PASS:", output_path, saved.sheetnames)
    print("CHECKPOINT 1/1 PASS: 미승인 차단 + 승인 후 Excel")
    download_artifact(output_path)
else:
    print(
        "Excel 생성 차단: 검증 오류를 최종 앱에서 수정한 뒤 "
        "다운로드하세요."
    )

complete_lab_step(9, 12, '`REVIEWED_APPROVED PASS`와 Excel 파일 경로를 확인합니다.')


## 최종 앱: 업로드부터 Excel 다운로드까지 한 화면으로 연결

아래 셀은 앞 교시의 기능을 하나의 실행 가능한 앱으로 묶습니다.
앱에서는 OCR/VLM 경로 선택, 원문·JSON 확인, 상호명·날짜·총액·품목
수정, 재검증, 사람 승인, Excel 다운로드를 순서대로 수행합니다.


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `FINAL_APP_SOURCE_PATHS`는 최종 앱에 필요한 파일 목록입니다. `load_course_assets()`로 파일을 받아 폴더에
# 저장한 뒤 `make_archive()`가 ZIP으로 묶습니다.
# ──────────────────────────────────────────────────────────
show_lab_step(10, 12, '최종 앱 묶음 생성', '업로드부터 Excel 다운로드까지 연결된 앱을 ZIP으로 묶습니다.', '최종 앱 경로와 ZIP 파일 경로가 표시되어야 합니다.', '`FINAL_APP_SOURCE_PATHS`는 최종 앱에 필요한 파일 목록입니다. `load_course_assets()`로 파일을 받아 폴더에 저장한 뒤 `make_archive()`가 ZIP으로 묶습니다.', 'none')

import shutil

FINAL_APP_SOURCE_PATHS = ['app.py', 'src/__init__.py', 'src/clean.py', 'src/export.py', 'src/extract.py', 'src/ocr.py', 'src/pipeline.py', 'src/sample_data.py', 'src/validate.py', 'src/vlm.py']
final_app_assets = load_course_assets(*FINAL_APP_SOURCE_PATHS)

final_app_dir = OUTPUT_DIR / "final_document_ai_app"
for relative_path in FINAL_APP_SOURCE_PATHS:
    target = final_app_dir / relative_path
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_bytes(final_app_assets[relative_path])

final_app_path = final_app_dir / "app.py"
archive_base = OUTPUT_DIR / "final_document_ai_app"
archive_path = Path(
    shutil.make_archive(
        str(archive_base),
        "zip",
        root_dir=final_app_dir,
    )
)
print("최종 앱:", final_app_path)
print("앱 전체 코드:", archive_path)

complete_lab_step(10, 12, '최종 앱 경로와 ZIP 파일 경로가 표시되어야 합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# `AppTest`가 오류값 저장 차단, 정상값 복구, 사람 승인, Excel 다운로드를 버튼 조작으로 검사합니다. 최종 통합 테스트입니다.
# ──────────────────────────────────────────────────────────
show_lab_step(11, 12, '최종 앱 자동 동작 검사', '수정·재검증·승인 전 차단·승인 후 다운로드를 검사합니다.', '`FINAL APP PASS`가 표시되어야 합니다.', '`AppTest`가 오류값 저장 차단, 정상값 복구, 사람 승인, Excel 다운로드를 버튼 조작으로 검사합니다. 최종 통합 테스트입니다.', 'none')

import sys
from streamlit.testing.v1 import AppTest

sys.path.insert(0, str(final_app_dir))
final_test = AppTest.from_file(str(final_app_path)).run(timeout=30)
assert not final_test.exception
final_test.button(key="run_sample").click().run(timeout=30)
assert not final_test.exception
assert any(
    "원본 대조 후 수정" in item.value
    for item in final_test.subheader
)
assert len(final_test.get("download_button")) == 0
final_test.checkbox(key="review_complete").check().run(timeout=30)
assert not final_test.exception
assert len(final_test.get("download_button")) == 1
print("FINAL APP PASS: 수정 표·재검증·승인·Excel 다운로드")
download_artifact(archive_path)

complete_lab_step(11, 12, '`FINAL APP PASS`가 표시되어야 합니다.')


In [ ]:
# ── 코드 읽기 ─────────────────────────────────────────────
# 최종 Streamlit 앱을 Colab iframe으로 열어 전체 흐름을 직접 조작합니다. 자동검증에서는 서버 화면만 생략합니다.
# ──────────────────────────────────────────────────────────
show_lab_step(12, 12, '최종 앱 직접 조작', 'Colab 안에서 전체 Document AI 흐름을 직접 수행합니다.', '앱 화면 또는 검증 모드 생략 안내를 확인합니다.', '최종 Streamlit 앱을 Colab iframe으로 열어 전체 흐름을 직접 조작합니다. 자동검증에서는 서버 화면만 생략합니다.', 'none')

# 선택 실습 · 녹화에서는 이 셀로 실제 화면을 엽니다.
# AppTest가 필수 검증이며, 미리보기에는 공개 비식별 샘플만 사용합니다.
if not VALIDATION_MODE:
    import subprocess
    import time
    import urllib.request

    preview_process = subprocess.Popen(
        [
            sys.executable, "-m", "streamlit", "run",
            str(final_app_path),
            "--server.port", "8507",
            "--server.headless", "true",
            "--server.enableCORS", "false",
            "--server.enableXsrfProtection", "false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
    )
    for _ in range(20):
        try:
            urllib.request.urlopen(
                "http://127.0.0.1:8507/_stcore/health",
                timeout=1,
            )
            break
        except Exception:
            time.sleep(0.5)
    try:
        from google.colab import output
        print("아래 화면에서 직접 버튼과 입력값을 조작하세요.")
        output.serve_kernel_port_as_iframe(8507, height=760)
    except Exception as exc:
        print("Colab 미리보기를 열지 못했습니다:", exc)
        print("AppTest 결과와 app 파일로 계속합니다.")
else:
    print("검증 모드: 대화형 Streamlit 미리보기 생략")

complete_lab_step(12, 12, '앱 화면 또는 검증 모드 생략 안내를 확인합니다.')
